# COMM075 — Titanic: Data Preparation Pipeline
**Dataset:** https://www.kaggle.com/competitions/titanic  
**Methodology:** CRISP-DM  

**Files required** (place in the same directory as this notebook):
- `train.csv` — 891 labelled rows used for training & validation
- `test.csv` — 418 unlabelled rows used for final predictions
- `gender_submission.csv` — sample submission showing expected output format

This notebook covers:
1. Imports
2. Data Loading & Initial Inspection
3. Data Quality Assessment
4. Feature Engineering
5. Missing Value Handling
6. Outlier Detection
7. Categorical Feature Encoding
8. Re-split into Train & Test
9. Feature Matrix & Scaling
10. Train / Validation Split
11. Exploratory Data Visualisation
12. Save Prepared Data
13. Summary Report

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import re

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
os.makedirs("output_plots_titanic", exist_ok=True)

print("Libraries loaded successfully.")

## 2. Data Loading & Initial Inspection
Load all three files and inspect the training set shape, column names, data types and first rows.

> **Note on `gender_submission.csv`:** This is a naive baseline submission provided by Kaggle — it predicts all females survive and all males do not. It shows the expected submission format (PassengerId + Survived), not ground truth labels for `test.csv`.

In [ ]:
train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")
gender_df = pd.read_csv("gender_submission.csv")

print(f"train.csv shape:             {train_df.shape}")
print(f"test.csv shape:              {test_df.shape}")
print(f"gender_submission.csv shape: {gender_df.shape}")

In [ ]:
# Column names and data types
print("train.csv columns and dtypes:")
print(train_df.dtypes)

In [ ]:
# First 5 rows
train_df.head()

In [ ]:
# Gender submission format (naive baseline)
print("gender_submission.csv — expected submission format:")
gender_df.head()

## 3. Data Quality Assessment
Identify missing values, duplicates and class balance in both train and test sets.

Key findings:
| Feature | Missing in train | Missing in test | Strategy |
|---|---|---|---|
| `Age` | ~20% | ~21% | Median imputation by Title & Pclass |
| `Cabin` | ~77% | ~78% | Convert to binary `CabinKnown` flag |
| `Embarked` | 2 rows | 0 | Mode imputation |
| `Fare` | 0 | 1 row | Median imputation by Pclass |

In [ ]:
def quality_report(df, name):
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    report = pd.DataFrame({
        "Missing Count": missing,
        "Missing %": missing_pct.round(2),
        "Dtype": df.dtypes
    })
    print(f"[{name}] Quality Report:")
    print(report.to_string())
    print(f"Duplicates: {df.duplicated().sum()}\n")

quality_report(train_df, "train.csv")
quality_report(test_df,  "test.csv")

In [ ]:
# Target class balance
print("Survival distribution (train.csv):")
print(train_df["Survived"].value_counts())
print()
print(train_df["Survived"].value_counts(normalize=True).round(3))

In [ ]:
# Summary statistics
train_df.describe()

## 4. Feature Engineering
New features are derived **before** imputation so they can inform the imputation strategy (e.g. Title helps predict Age).

| New Feature | Derived from | Rationale |
|---|---|---|
| `Title` | `Name` | Encodes gender, age group and social status (Mr, Mrs, Miss, Master, Rare) |
| `FamilySize` | `SibSp + Parch + 1` | Solo travellers vs. small/large families had different survival rates |
| `IsAlone` | `FamilySize == 1` | Binary flag — travelling alone is a strong signal |
| `CabinKnown` | `Cabin` | 77% missing; presence of a cabin record correlates with higher class |
| `FareBand` | `Fare` | Quartile bins of fare — created after imputation |
| `AgeBand` | `Age` | 5 life-stage bins — created after imputation |

In [ ]:
# Combine train and test for consistent encoding
# (we re-split at the end using the '_set' marker column)
train_df["_set"] = "train"
test_df["_set"]  = "test"
test_df["Survived"] = np.nan   # placeholder so columns match
combined = pd.concat([train_df, test_df], ignore_index=True)

print(f"Combined shape: {combined.shape}")

In [ ]:
# 4a — Extract Title from Name
combined["Title"] = combined["Name"].apply(
    lambda x: re.search(r",\s*([^.]+)\.", x).group(1).strip()
    if re.search(r",\s*([^.]+)\.", x) else "Unknown"
)

print("Raw titles found:")
print(combined["Title"].value_counts().to_dict())

In [ ]:
# Consolidate rare titles into broader groups
title_map = {
    "Mr": "Mr",     "Miss": "Miss",   "Mrs": "Mrs",   "Master": "Master",
    "Dr": "Rare",   "Rev": "Rare",    "Col": "Rare",  "Major": "Rare",
    "Mlle": "Miss", "Mme": "Mrs",     "Ms": "Miss",
    "Capt": "Rare", "Don": "Rare",    "Sir": "Rare",
    "Countess": "Rare", "Jonkheer": "Rare", "Dona": "Rare", "Lady": "Rare"
}
combined["Title"] = combined["Title"].map(title_map).fillna("Rare")

print("Consolidated titles:")
print(combined["Title"].value_counts().to_dict())

In [ ]:
# 4b — Family Size & IsAlone
combined["FamilySize"] = combined["SibSp"] + combined["Parch"] + 1
combined["IsAlone"]    = (combined["FamilySize"] == 1).astype(int)

print(f"FamilySize range: {combined['FamilySize'].min()} – {combined['FamilySize'].max()}")
print(f"Travelling alone: {combined['IsAlone'].sum()} passengers")

In [ ]:
# 4c — CabinKnown (binary flag)
# 77% of Cabin values are missing — too sparse to impute reliably.
# Having a recorded cabin correlates with higher-class (wealthier) passengers.
combined["CabinKnown"] = combined["Cabin"].notnull().astype(int)

print("CabinKnown distribution:")
print(combined["CabinKnown"].value_counts().to_dict())

## 5. Missing Value Handling
Each feature uses a tailored imputation strategy rather than a single global approach.

In [ ]:
# 5a — Age: median imputation grouped by Title & Pclass
# Justification: a 'Master' (young boy) has a very different age profile than a 'Mr'.
# Using group medians preserves this demographic structure.
age_medians = combined.groupby(["Title", "Pclass"])["Age"].median()
print("Age medians by Title & Pclass:")
print(age_medians)

In [ ]:
def impute_age(row):
    if pd.isnull(row["Age"]):
        try:
            return age_medians.loc[(row["Title"], row["Pclass"])]
        except KeyError:
            return combined["Age"].median()
    return row["Age"]

combined["Age"] = combined.apply(impute_age, axis=1)
print(f"Age missing after imputation: {combined['Age'].isnull().sum()}")

In [ ]:
# 5b — Embarked: mode imputation (only 2 rows missing)
embarked_mode = combined["Embarked"].mode()[0]
combined["Embarked"].fillna(embarked_mode, inplace=True)
print(f"Embarked filled with mode: '{embarked_mode}'")

In [ ]:
# 5c — Fare: median by Pclass (only 1 row missing, in test set)
fare_medians = combined.groupby("Pclass")["Fare"].median()
combined["Fare"] = combined.apply(
    lambda row: fare_medians[row["Pclass"]] if pd.isnull(row["Fare"]) else row["Fare"],
    axis=1
)
print(f"Fare missing after imputation: {combined['Fare'].isnull().sum()}")

In [ ]:
# 5d — Create AgeBand and FareBand now that imputation is complete
combined["FareBand"] = pd.qcut(combined["Fare"], q=4, labels=[0, 1, 2, 3]).astype(int)
combined["AgeBand"]  = pd.cut(combined["Age"],
                               bins=[0, 12, 18, 35, 60, 120],
                               labels=[0, 1, 2, 3, 4]).astype(int)

print("FareBand value counts:")
print(combined["FareBand"].value_counts().sort_index())
print("\nAgeBand value counts:")
print(combined["AgeBand"].value_counts().sort_index())

In [ ]:
# Drop columns we no longer need
combined.drop(columns=["Name", "Ticket", "Cabin"], inplace=True)
print(f"Dropped: Name, Ticket, Cabin")
print(f"Remaining columns: {list(combined.columns)}")

In [ ]:
# Verify no remaining missingness (excluding Survived which is NaN for test set)
missing_check = combined.drop(columns=["Survived"]).isnull().sum()
remaining = missing_check[missing_check > 0]
if len(remaining) == 0:
    print("✓ No missing values remaining in any feature")
else:
    print(remaining)

## 6. Outlier Detection
Use the IQR method on continuous features. `Fare` has extreme outliers (first-class suites) — we apply a **log1p transform**.

In [ ]:
train_only = combined[combined["_set"] == "train"]
num_cols   = ["Age", "Fare", "FamilySize"]

print(f"{'Feature':<15} {'Q1':>6} {'Q3':>6} {'IQR':>6}  {'Lower':>7} {'Upper':>7}  {'Outliers':>8}")
print("-" * 65)
for col in num_cols:
    Q1 = train_only[col].quantile(0.25)
    Q3 = train_only[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((train_only[col] < lower) | (train_only[col] > upper)).sum()
    print(f"{col:<15} {Q1:>6.1f} {Q3:>6.1f} {IQR:>6.1f}  {lower:>7.1f} {upper:>7.1f}  {n_out:>8}")

In [ ]:
# Apply log1p to Fare (right-skewed, extreme first-class outliers)
combined["Fare"] = np.log1p(combined["Fare"])
print("log1p transform applied to Fare.")

## 7. Categorical Feature Encoding

| Feature | Strategy | Reason |
|---|---|---|
| `Sex` | Binary map | male=0, female=1 |
| `Embarked` | Ordinal map | S=0, C=1, Q=2 (3 categories, low cardinality) |
| `Title` | LabelEncoder | 5 consolidated categories |

In [ ]:
# Sex — binary map
combined["Sex"] = combined["Sex"].map({"male": 0, "female": 1})
print("Sex encoded: male=0, female=1")

In [ ]:
# Embarked — ordinal map
combined["Embarked"] = combined["Embarked"].map({"S": 0, "C": 1, "Q": 2})
print("Embarked encoded: S=0, C=1, Q=2")

In [ ]:
# Title — LabelEncoder
le = LabelEncoder()
combined["Title"] = le.fit_transform(combined["Title"])
print(f"Title classes: {dict(zip(le.classes_, le.transform(le.classes_)))}")

In [ ]:
print(f"\nFinal combined columns ({len(combined.columns)}):")
print(list(combined.columns))

## 8. Re-split into Train & Test
Separate the combined dataframe back into labelled training data and unlabelled test data.

In [ ]:
train_clean = combined[combined["_set"] == "train"].drop(columns=["_set"]).copy()
test_clean  = combined[combined["_set"] == "test"].drop(columns=["_set", "Survived"]).copy()

print(f"train_clean shape: {train_clean.shape}")
print(f"test_clean shape:  {test_clean.shape}")

In [ ]:
feature_cols = [
    "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare",
    "Embarked", "Title", "FamilySize", "IsAlone",
    "CabinKnown", "FareBand", "AgeBand"
]

X            = train_clean[feature_cols]
y            = train_clean["Survived"].astype(int)
X_final_test = test_clean[feature_cols]   # unlabelled — for final submission

print(f"X shape:            {X.shape}")
print(f"y shape:            {y.shape}")
print(f"X_final_test shape: {X_final_test.shape}")

## 9. Feature Matrix & Scaling
Apply **StandardScaler** (zero mean, unit variance).  
The scaler is fit on training data only, then applied to both train and test to avoid data leakage.

In [ ]:
scaler = StandardScaler()
X_scaled      = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols)
X_test_scaled = pd.DataFrame(scaler.transform(X_final_test), columns=feature_cols)

print(f"X_scaled shape:      {X_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")
print(f"\nMean (should be ≈ 0):")
print(X_scaled.mean().round(3))
print(f"\nStd (should be ≈ 1):")
print(X_scaled.std().round(3))

## 10. Train / Validation Split
Since `test.csv` has no labels (competition format), we split `train.csv` **80/20** for model evaluation.  
Stratified to preserve the ~38% survival rate in both sets.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape}  |  y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}    |  y_val:   {y_val.shape}")
print(f"\nTraining class balance:")
print(y_train.value_counts(normalize=True).round(3))
print(f"\nValidation class balance:")
print(y_val.value_counts(normalize=True).round(3))

## 11. Exploratory Data Visualisation
Nine charts exploring survival rates by key features and the distributions of engineered variables.

In [ ]:
sns.set_theme(style="whitegrid")
raw_train = train_clean.copy()  # use cleaned but pre-scaled version for interpretable plots

In [ ]:
# 11a — Overall survival distribution
fig, ax = plt.subplots(figsize=(5, 4))
counts = y.value_counts().sort_index()
ax.bar(["Did not survive (0)", "Survived (1)"], counts,
       color=["#C44E52", "#4C72B0"], edgecolor="white", linewidth=1.2)
for i, (label, count) in enumerate(zip(["Did not survive", "Survived"], counts)):
    ax.text(i, count + 5, f"{count} ({count/len(y)*100:.1f}%)", ha="center", fontsize=10)
ax.set_title("Overall Survival Distribution", fontweight="bold")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig("output_plots_titanic/01_survival_distribution.png", dpi=150)
plt.show()

In [ ]:
# 11b — Survival rate by Sex
fig, ax = plt.subplots(figsize=(6, 4))
sex_survival = raw_train.groupby("Sex")["Survived"].mean()
ax.bar(["Male (0)", "Female (1)"], sex_survival.values,
       color=["#4C72B0", "#DD8452"], edgecolor="white")
for i, v in enumerate(sex_survival.values):
    ax.text(i, v + 0.02, f"{v:.2%}", ha="center")
ax.set_title("Survival Rate by Sex", fontweight="bold")
ax.set_ylabel("Survival Rate")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig("output_plots_titanic/02_survival_by_sex.png", dpi=150)
plt.show()

In [ ]:
# 11c — Survival rate by Passenger Class
fig, ax = plt.subplots(figsize=(6, 4))
pclass_survival = raw_train.groupby("Pclass")["Survived"].mean()
ax.bar([f"Class {i}" for i in pclass_survival.index], pclass_survival.values,
       color=["#4C72B0", "#55A868", "#C44E52"], edgecolor="white")
for i, v in enumerate(pclass_survival.values):
    ax.text(i, v + 0.02, f"{v:.2%}", ha="center")
ax.set_title("Survival Rate by Passenger Class", fontweight="bold")
ax.set_ylabel("Survival Rate")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig("output_plots_titanic/03_survival_by_pclass.png", dpi=150)
plt.show()

In [ ]:
# 11d — Age distribution by survival
fig, ax = plt.subplots(figsize=(9, 4))
for label, grp in raw_train.groupby("Survived"):
    ax.hist(grp["Age"], bins=30, alpha=0.6, density=True, label=f"Survived={int(label)}")
ax.set_title("Age Distribution by Survival", fontweight="bold")
ax.set_xlabel("Age")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.savefig("output_plots_titanic/04_age_by_survival.png", dpi=150)
plt.show()

In [ ]:
# 11e — Family size vs survival rate
fig, ax = plt.subplots(figsize=(8, 4))
fam_survival = raw_train.groupby("FamilySize")["Survived"].mean()
ax.bar(fam_survival.index, fam_survival.values,
       color=sns.color_palette("Set2", len(fam_survival)), edgecolor="white")
ax.set_title("Survival Rate by Family Size", fontweight="bold")
ax.set_xlabel("Family Size (1 = travelling alone)")
ax.set_ylabel("Survival Rate")
ax.set_xticks(fam_survival.index)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig("output_plots_titanic/05_survival_by_familysize.png", dpi=150)
plt.show()

In [ ]:
# 11f — Fare distribution by survival (after log1p)
fig, ax = plt.subplots(figsize=(8, 4))
for label, grp in raw_train.groupby("Survived"):
    ax.hist(grp["Fare"], bins=40, alpha=0.6, density=True, label=f"Survived={int(label)}")
ax.set_title("Fare Distribution by Survival (log1p scaled)", fontweight="bold")
ax.set_xlabel("log1p(Fare)")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.savefig("output_plots_titanic/06_fare_by_survival.png", dpi=150)
plt.show()

In [ ]:
# 11g — Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 7))
corr = raw_train[feature_cols + ["Survived"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=ax, linewidths=0.5, annot_kws={"size": 7})
ax.set_title("Correlation Matrix — All Features + Target", fontweight="bold")
plt.tight_layout()
plt.savefig("output_plots_titanic/07_correlation_heatmap.png", dpi=150)
plt.show()

In [ ]:
# 11h — Survival rate by encoded Title
fig, ax = plt.subplots(figsize=(7, 4))
title_survival = raw_train.groupby("Title")["Survived"].mean().sort_values(ascending=False)
ax.bar(title_survival.index.astype(str), title_survival.values,
       color=sns.color_palette("Set1", len(title_survival)), edgecolor="white")
ax.set_title("Survival Rate by Encoded Title", fontweight="bold")
ax.set_xlabel("Title (encoded integer)")
ax.set_ylabel("Survival Rate")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig("output_plots_titanic/08_survival_by_title.png", dpi=150)
plt.show()

In [ ]:
# 11i — Survival rate by Cabin record presence
fig, ax = plt.subplots(figsize=(5, 4))
cabin_survival = raw_train.groupby("CabinKnown")["Survived"].mean()
ax.bar(["No Cabin Record", "Cabin Known"], cabin_survival.values,
       color=["#C44E52", "#4C72B0"], edgecolor="white")
for i, v in enumerate(cabin_survival.values):
    ax.text(i, v + 0.02, f"{v:.2%}", ha="center")
ax.set_title("Survival Rate by Cabin Record", fontweight="bold")
ax.set_ylabel("Survival Rate")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig("output_plots_titanic/09_survival_by_cabin.png", dpi=150)
plt.show()

## 12. Save Prepared Data
Export all splits as CSV files ready for model development.

| File | Description |
|---|---|
| `titanic_X_train.csv` | Training features (scaled, label-encoded) |
| `titanic_X_val.csv` | Validation features |
| `titanic_y_train.csv` | Training labels |
| `titanic_y_val.csv` | Validation labels |
| `titanic_X_test.csv` | Unlabelled test features (for final submission) |
| `titanic_passenger_ids.csv` | PassengerId column for submission file |

In [ ]:
X_train.to_csv("titanic_X_train.csv", index=False)
X_val.to_csv("titanic_X_val.csv", index=False)
y_train.to_csv("titanic_y_train.csv", index=False)
y_val.to_csv("titanic_y_val.csv", index=False)
X_test_scaled.to_csv("titanic_X_test.csv", index=False)
test_clean[["PassengerId"]].to_csv("titanic_passenger_ids.csv", index=False)

print("Saved:")
print("  titanic_X_train.csv, titanic_X_val.csv")
print("  titanic_y_train.csv, titanic_y_val.csv")
print("  titanic_X_test.csv  (unlabelled — for final submission)")
print("  titanic_passenger_ids.csv")

## 13. Summary Report

In [ ]:
print("=" * 60)
print("DATA PREPARATION SUMMARY — TITANIC")
print("=" * 60)
print(f"  train.csv original shape:        891 rows × 12 cols")
print(f"  test.csv original shape:         418 rows × 11 cols")
print(f"  Features dropped:                Name, Ticket, Cabin")
print(f"  Features engineered:             Title, FamilySize, IsAlone,")
print(f"                                   CabinKnown, FareBand, AgeBand")
print(f"  Missing: Age (~20%)              Median by Title & Pclass")
print(f"  Missing: Embarked (2 rows)       Mode imputation ('{embarked_mode}')")
print(f"  Missing: Fare (1 row)            Median by Pclass")
print(f"  Missing: Cabin (77%)             Converted to binary CabinKnown")
print(f"  Skew correction:                 log1p on Fare")
print(f"  Encoding:                        Sex (binary), Embarked (ordinal),")
print(f"                                   Title (LabelEncoder)")
print(f"  Scaling:                         StandardScaler (fit on train only)")
print(f"  Train/validation split:          80% / 20% stratified")
print(f"  X_train: {X_train.shape}  |  X_val: {X_val.shape}")
print(f"  Class balance (train):           {y_train.value_counts(normalize=True).round(3).to_dict()}")
print(f"  Final test set:                  {X_test_scaled.shape} (no labels)")
print("=" * 60)

In [ ]:
# Sanity check on X_train
print("X_train — first 5 rows:")
display(X_train.head())
print("\nX_train — describe:")
display(X_train.describe())